In [1]:
import pandas as pd
import numpy as np
import os
import math
from pathlib import Path
from matplotlib import pyplot as plt
idx = pd.IndexSlice
import xarray as xr
from read_mym import read_mym_df

from constants_BUMA import(
    SCENARIO_SELECT, FILE_ADDITION,
    REGIONS, REGIONS_RANGE,
    START_YEAR, END_YEAR, HIST_YEAR, YEARS, YEAR_LIST_SVA,
    INFLATION, 
    FLAG_ALPHA, FLAG_EXPDEC, FLAG_NORMAL,
    LOWCOMM,
    GOMPERTZ_EXPDEC,
    MINIMUM_COM
)


In [2]:
base_directory = Path("..", "IMAGE-Mat_old_version", "IMAGE-Mat", "BUMA")
database_directory = base_directory / "files_DB" / SCENARIO_SELECT
image_directory = base_directory / "files_IMAGE" / SCENARIO_SELECT
assert database_directory.is_dir(), database_directory
assert image_directory.is_dir()

In [3]:
# Pop; unit: million of people; meaning: global population (over time, by region)             
population: pd.DataFrame = pd.read_csv(image_directory.joinpath('pop.csv'), index_col = [0]) 
# rurpop; unit: %; meaning: the share of people living in rural areas (over time, by region)
rural_population: pd.DataFrame = pd.read_csv(image_directory. joinpath('rurpop.csv'), index_col = [0])
# load historic population development
historic_population = pd.read_csv(base_directory / 'files_initial_stock' /'hist_pop.csv', index_col = [0])  
# initial population as a percentage of the 1970 population; unit: %; according to the Maddison Project Database (MPD) 2018 (Groningen University)


In [4]:
# Interpolate population and rural population data (fills in missing years with cubic interpolation)
rural_population = rural_population.reindex(YEARS).interpolate(method='cubic')
population = population.reindex(YEARS).interpolate(method='cubic')
# Remove 1st year, to ensure same Table size as floorspace data (from 1971)
#population = population.iloc[1:]
#rural_population = rural_population.iloc[1:]
#TODO unsure why first vakue is removed

#pre-calculate urban population
urban_population = 1 - rural_population    
# urban population is 1 - the fraction of people living in rural areas (rural_population)


In [5]:
# Deriving historic population tail based on fraction for 1970
population_1970 = population.loc[1970]
historic_population = historic_population.multiply(population_1970, axis=1)
population = pd.concat([historic_population.loc[:1969], population])

In [6]:
# Get the growth or rural population by year (average for the first 10 years of IMAGE data)
rural_population_trend = ((rural_population.loc[1980]/rural_population.loc[1970])/10)*100
maximum_rural_population = rural_population.values.max()

In [7]:
population = population.rename_axis(index = "Year", columns = "Region")

In [8]:
population_array = xr.DataArray(
    data=population.values,               # Data values from the DataFrame
    dims=["Year", "Region"],       # Names for the two dimensions
    coords={"Year": population.index,      # Year coordinates from the DataFrame index
            "Region": population.columns}  # Region coordinates from the DataFrame columns
    )

In [13]:
far_start_year = 1721
start_year = 1820
end_year = 1970

interpol_population_1721_1820 = np.linspace(0, population_array[0, :].values, num=(1+start_year - far_start_year), endpoint=False)[1:]
interpol_population_1721_1820_xr = xr.DataArray(interpol_population_1721_1820,
                                          dims=population_array.dims,
                                          coords={
                                              "Year": np.arange(far_start_year, start_year),
                                              "Region": population_array.coords["Region"]
                                          })

In [14]:
population_xr = xr.concat((interpol_population_1721_1820_xr, population_array), dim="Year")

In [18]:
population_xr

<xarray.DataArray (Year: 340, Region: 26)> Size: 71kB
array([[8.04857371e-03, 1.02020353e-01, 6.49401441e-02, ...,
        4.81580833e-03, 3.48787298e-01, 9.66412036e-02],
       [1.60971474e-02, 2.04040705e-01, 1.29880288e-01, ...,
        9.63161666e-03, 6.97574595e-01, 1.93282407e-01],
       [2.41457211e-02, 3.06061058e-01, 1.94820432e-01, ...,
        1.44474250e-02, 1.04636189e+00, 2.89923611e-01],
       ...,
       [5.00346937e+01, 4.13421703e+02, 1.57426121e+02, ...,
        4.83825966e+01, 6.59012994e+02, 3.37412746e+02],
       [5.03156408e+01, 4.15282729e+02, 1.57447121e+02, ...,
        4.86897347e+01, 6.61447460e+02, 3.40567607e+02],
       [5.05944800e+01, 4.17148800e+02, 1.57433800e+02, ...,
        4.89911500e+01, 6.63727400e+02, 3.43669300e+02]])
Coordinates:
  * Year     (Year) int64 3kB 1721 1722 1723 1724 1725 ... 2057 2058 2059 2060
  * Region   (Region) object 208B '1' '2' '3' '4' '5' ... '23' '24' '25' '26'